# Laboratorio de detección y comparación facial con Amazon Rekognition

En este laboratorio, usaremos Amazon Rekognition para realizar detección y comparación facial entre dos grupos de imágenes de diferentes personas.

El objetivo es determinar a qué grupo pertenece un rostro objetivo, basándonos en la cantidad de coincidencias con imágenes de referencia de cada grupo.

Los pasos generales que realizará en este laboratorio son los siguientes:

1. Crear una colección facial en Amazon Rekognition.

2. Cargar múltiples imágenes de dos personas distintas y etiquetarlas como `Persona A` y `Persona B`.

3. Agregar estas imágenes a la colección con sus respectivos identificadores (`ExternalImageId`).

4. Buscar coincidencias en la colección para un rostro objetivo.

5. Comparar las coincidencias encontradas, agrupando los resultados por `ExternalImageId`.

6. Determinar a qué persona pertenece el rostro objetivo en función del número de coincidencias y su similitud.

7. Visualizar las imágenes procesadas, incluyendo los cuadros delimitadores de los rostros detectados.

8. Eliminar la colección para limpiar los recursos al final del laboratorio.

---

Este laboratorio demostrará cómo agrupar resultados de detección facial utilizando `ExternalImageId` y cómo tomar decisiones basadas en estas agrupaciones. También aprenderá cómo Amazon Rekognition procesa rostros a partir de múltiples imágenes de referencia.


## Importación de paquetes de Python

Empiece por importar los paquetes Python que necesita.

En el siguiente código:

- *matplotlib* proporciona funciones de trazado
- *skimage* representa scikit-image, que proporciona varias herramientas útiles de manipulación de imágenes
- *boto3* representa el AWS SDK para Python (Boto3), que es la biblioteca de Python para AWS
- *numpy* representa NumPy, que es una biblioteca para manipular datos
- *PIL* representa la biblioteca de imágenes de Python, que contiene un conjunto de herramientas para dibujar imágenes


In [ ]:
from skimage import io
# Importa el módulo `io` de la librería `skimage` para manejar imágenes (lectura y escritura).
import os

from skimage.transform import rescale
# Importa la función `rescale` de `skimage.transform`, que permite redimensionar imágenes manteniendo la proporción.

from matplotlib import pyplot as plt
# Importa el módulo `pyplot` de `matplotlib`, usado para crear gráficos y mostrar imágenes.

import boto3
# Importa la librería `boto3`, que es el SDK de Python para interactuar con servicios de AWS (como Rekognition).

import numpy as np
# Importa la librería `numpy`, que se utiliza para realizar operaciones matemáticas y manejar arrays de manera eficiente.

from PIL import Image, ImageDraw, ImageColor, ImageOps
# Importa clases de la librería `Pillow` (PIL):
# - `Image`: para cargar y manejar imágenes.
# - `ImageDraw`: para dibujar formas (como rectángulos) sobre imágenes.
# - `ImageColor`: para manejar colores en diferentes formatos.
# - `ImageOps`: para realizar operaciones como bordes y reflejos en imágenes.


## Tarea 1: crear una colección

En esta tarea, creará una colección en Amazon Rekognition.

Solo debe ejecutar este paso una vez.


In [ ]:
client = boto3.client('rekognition')
# Crea un cliente para interactuar con el servicio de Amazon Rekognition usando Boto3.
# Este cliente permite llamar a las APIs de Rekognition para realizar operaciones como crear colecciones, detectar rostros, etc.

collection_id = 'Collection'
# Define un identificador único para la colección de imágenes que deseas crear.
# Este identificador será el nombre de tu colección en Rekognition.
# NOTA DE JANU: Si queremos cambiar la collection debemos cambiar el ID

print(f"Intentando crear o verificar la colección con ID: {collection_id}")
# Imprime el nombre de la colección que estamos manejando, para que los alumnos sepan cuál estamos procesando.

# Verificar si la colección ya existe
existing_collections = client.list_collections()['CollectionIds']
# Llama a la API `list_collections` de Rekognition para obtener una lista de todas las colecciones existentes.
# La respuesta es un diccionario, y `CollectionIds` contiene una lista de los nombres de las colecciones.

if collection_id in existing_collections:
    # Comprueba si el ID de la colección que deseas crear ya está en la lista de colecciones existentes.
    print(f"La colección '{collection_id}' ya existe.")
    # Si la colección ya existe, se imprime un mensaje y no se intenta crearla nuevamente.
else:
    # Crear la colección si no existe
    response = client.create_collection(CollectionId=collection_id)
    # Llama a la API `create_collection` para crear una nueva colección con el ID especificado.
    # Si la colección se crea exitosamente, AWS devuelve un ARN (Amazon Resource Name) y un código de estado.

    print('Collection ARN: ' + response['CollectionArn'])
    # Imprime el ARN de la colección recién creada.
    # El ARN es un identificador único que AWS asigna a los recursos creados.

    print('Status Code: ' + str(response['StatusCode']))
    # Imprime el código de estado de la operación. Un código `200` indica que la colección se creó exitosamente.

    print('La colección se creó exitosamente.')
    # Indica que el proceso de creación de la colección ha finalizado.

print('Done...')
# Muestra un mensaje general indicando que el script terminó su ejecución.



## Tarea 2: cargar una imagen para buscar

Use la imagen de muestra provista, con el nombre *mum.jpg*, y cárguela en este cuaderno.

Luego, mire la imagen ejecutando la celda.


<h1 style="color:purple;">NOTAS DE JANU</h1>

Si cambiamos el filename podemos elegir cualquier imagen como modelo. En este caso estoy usando model.png con una imagen mia para demo


In [ ]:
# Define el path actual y la carpeta de modelos
current_directory = os.getcwd()
models_folder = os.path.join(current_directory, "models")

# Recorrer todas las subcarpetas en "models" (una subcarpeta por persona)
for person_folder in os.listdir(models_folder):
    person_path = os.path.join(models_folder, person_folder)
    # Asegúrate de que solo procese carpetas
    if os.path.isdir(person_path):
        print(f"Procesando imágenes de: {person_folder}")
        # Imprime el nombre de la subcarpeta que se está procesando.

        # Lista todos los archivos en la subcarpeta
        model_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in model_files:
            filepath = os.path.join(person_path, filename)
            # Construye la ruta completa para cada archivo de imagen.

            print(f"Cargando la imagen: {filename} de la carpeta {person_folder}")
            # Imprime el nombre de la imagen que se está cargando.

            faceimage = io.imread(filepath)
            # Carga la imagen desde el archivo usando `skimage.io`.

            plt.imshow(faceimage)
            plt.title(f"{person_folder}: {filename}")
            plt.show()
            # Muestra la imagen utilizando Matplotlib.



Asegúrese de que el tamaño de la imagen sea menos de 4096 x 4096 píxeles. Si la imagen tiene un tamaño mayor, debe ajustarlo con el siguiente código:

`faceimage = rescale(faceimage, 0.50, mode='constant')`

**Nota:** El valor numérico representa el factor según el que se debe escalar. Un valor de *0.5* escalará la imagen a un 50 por ciento de la original.

Cuando se ajuste el tamaño de la imagen, guarde el archivo.

`io.imsave(filename, faceimage)`

**Sugerencia:** Debe copiar el código en una celda de código y ejecutarla.


## Tarea 3: agregar la imagen a la colección

Agregue la imagen a la colección que creó anteriormente.


In [ ]:
# Define el path actual y la carpeta de modelos
current_directory = os.getcwd()
models_folder = os.path.join(current_directory, "models")

# Recorrer todas las subcarpetas en "models" (una subcarpeta por persona)
for person_folder in os.listdir(models_folder):
    person_path = os.path.join(models_folder, person_folder)
    # Asegúrate de que solo procese carpetas
    if os.path.isdir(person_path):
        print(f"Procesando imágenes de: {person_folder}")
        # Usa el nombre de la subcarpeta como ExternalImageId
        externalimageid = person_folder

        # Lista todos los archivos en la subcarpeta
        model_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in model_files:
            filepath = os.path.join(person_path, filename)
            # Construye la ruta completa para cada archivo de imagen.

            print(f"Verificando o subiendo la imagen: {filename} de la carpeta {person_folder}")
            # Verificar si el rostro ya está indexado
            response = client.list_faces(CollectionId=collection_id)
            faces = response['Faces']
            found = False

            for face in faces:
                if face['ExternalImageId'] == externalimageid:
                    found = True
                    print(f"El rostro con ExternalImageId '{externalimageid}' ya está indexado con FaceId: {face['FaceId']}.")
                    break

            if not found:
                # Si el rostro no está indexado, procede a indexarlo.
                with open(filepath, 'rb') as fimage:
                    response = client.index_faces(
                        CollectionId=collection_id,
                        Image={'Bytes': fimage.read()},
                        ExternalImageId=externalimageid,
                        MaxFaces=1,
                        QualityFilter="AUTO",
                        DetectionAttributes=['ALL']
                    )

                print('Resultados para ' + filename)
                # Imprime los rostros indexados
                if 'FaceRecords' in response:
                    print('Faces indexed:')
                    for faceRecord in response['FaceRecords']:
                        print('  Face ID: ' + faceRecord['Face']['FaceId'])
                        print('  Location: {}'.format(faceRecord['Face']['BoundingBox']))
                else:
                    print('No se detectaron rostros en la imagen.')

                # Imprime rostros no indexados, si hay
                if 'UnindexedFaces' in response:
                    print('Faces not indexed:')
                    for unindexedFace in response['UnindexedFaces']:
                        print(' Location: {}'.format(unindexedFace['FaceDetail']['BoundingBox']))
                        print(' Reasons:')
                        for reason in unindexedFace['Reasons']:
                            print('   ' + reason)


## Tarea 4: ver el cuadro delimitador para el rostro detectado

Si se encuentra un rostro, los resultados deben incluir la ubicación del rostro que se detectó. Examine el cuadro delimitador en la imagen.

Para ello, use la biblioteca PIL, que importó anteriormente en este laboratorio. Al extraer BoundingBox, puede dibujar un conjunto de líneas alrededor de la imagen.


In [ ]:
# Define el path actual y la carpeta de modelos
current_directory = os.getcwd()
models_folder = os.path.join(current_directory, "models")

# Iterar sobre todas las subcarpetas en "models"
for person_folder in os.listdir(models_folder):
    person_path = os.path.join(models_folder, person_folder)
    # Asegúrate de que solo procese carpetas
    if os.path.isdir(person_path):
        externalimageid = person_folder
        # Usa el nombre de la subcarpeta como `ExternalImageId`.

        print(f"\nProcesando imágenes de: {person_folder}")
        # Lista todos los archivos en la subcarpeta
        model_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in model_files:
            filepath = os.path.join(person_path, filename)
            # Construye la ruta completa de la imagen actual.

            # Verificar si el rostro ya está indexado bajo este ExternalImageId
            response = client.list_faces(CollectionId=collection_id)
            faces = response['Faces']
            found = False

            for face in faces:
                if face['ExternalImageId'] == externalimageid:
                    # Si ya existe un rostro con este `ExternalImageId`, no indexamos esta imagen de nuevo
                    found = True
                    print(f"El rostro con ExternalImageId '{externalimageid}' ya está indexado con FaceId: {face['FaceId']}.")
                    break

            if not found:
                print(f"Indexando la imagen '{filename}' para ExternalImageId: {externalimageid}")
                with open(filepath, 'rb') as image_file:
                    response = client.index_faces(
                        CollectionId=collection_id,
                        Image={'Bytes': image_file.read()},
                        ExternalImageId=externalimageid,
                        MaxFaces=1,
                        QualityFilter="AUTO",
                        DetectionAttributes=['ALL']
                    )

                    if 'FaceRecords' in response:
                        for faceRecord in response['FaceRecords']:
                            print(f"Imagen '{filename}' indexada exitosamente con FaceId: {faceRecord['Face']['FaceId']}")
                    else:
                        print(f"No se detectaron rostros en la imagen '{filename}'.")

            # Mostrar la imagen cargada
            faceimage = io.imread(filepath)
            plt.imshow(faceimage)
            plt.title(f"{externalimageid}: {filename}")
            plt.show()


## Tarea 5: enumerar los rostros en la colección

Examine la imagen que tiene en la colección. 



In [ ]:
maxResults = 2
# Define la cantidad máxima de rostros que se recuperarán por solicitud para manejar la paginación.

faces_count = 0
# Inicializa un contador para rastrear la cantidad total de rostros en la colección.

tokens = True
# Variable de control para manejar la paginación de resultados.

response = client.list_faces(CollectionId=collection_id, MaxResults=maxResults)
# Realiza la primera solicitud para listar los rostros en la colección.

print('Rostros en la colección:', collection_id)
# Imprime un encabezado indicando que se mostrarán los rostros de la colección.

# Diccionario para agrupar rostros por `ExternalImageId`
faces_by_external_id = {}

while tokens:
    faces = response['Faces']
    # Extrae la lista de rostros de la respuesta actual.

    for face in faces:
        # Itera sobre cada rostro en la lista actual.
        external_id = face.get('ExternalImageId', 'Sin ID externo')
        # Obtiene el `ExternalImageId` del rostro o asigna "Sin ID externo" si no está presente.

        if external_id not in faces_by_external_id:
            faces_by_external_id[external_id] = []
        # Si el `ExternalImageId` no está en el diccionario, lo inicializa como una lista vacía.

        faces_by_external_id[external_id].append(face)
        # Agrega el rostro actual al grupo correspondiente al `ExternalImageId`.

        faces_count += 1
        # Incrementa el contador total de rostros.

    if 'NextToken' in response:
        # Verifica si hay más páginas de resultados.
        nextToken = response['NextToken']
        # Obtiene el token para la próxima página.
        response = client.list_faces(CollectionId=collection_id, NextToken=nextToken, MaxResults=maxResults)
        # Solicita la siguiente página de resultados.
    else:
        tokens = False
        # Si no hay más páginas, termina el bucle.

# Mostrar los resultados agrupados por `ExternalImageId`
for external_id, faces in faces_by_external_id.items():
    print(f"\nExternalImageId: {external_id}")
    # Imprime el identificador externo del grupo de rostros.
    for face in faces:
        print(f"  FaceId: {face['FaceId']}")
        print(f"  BoundingBox: {face['BoundingBox']}")
        print(f"  Similarity: {face.get('Similarity', 'N/A')}")  # Puede no estar presente en `list_faces`.

print(f'\nTotal de rostros en la colección: {faces_count}')
# Imprime el total de rostros encontrados después de recorrer todas las páginas.


## Tarea 6: encontrar un rostro usando la colección

En este paso, usará la colección para detectar un rostro en una imagen.

Use la imagen de muestra provista, con el nombre *target.jpg*, y cárguela en este cuaderno.



In [ ]:
# Obtiene el path actual donde se encuentra el script
current_directory = os.getcwd()

# Define la carpeta donde están las imágenes
images_folder = os.path.join(current_directory, "images")

# Lista todos los archivos en la carpeta "images"
image_files = [f for f in os.listdir(images_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
# Filtra los archivos para incluir solo imágenes con extensiones válidas.

# Procesa cada imagen en la carpeta
for targetfilename in image_files:
    targetfilepath = os.path.join(images_folder, targetfilename)
    # Construye la ruta completa para cada archivo de imagen.

    print(f"Procesando la imagen: {targetfilename}")
    # Imprime el nombre de la imagen que se está procesando.

    targetimage = Image.open(targetfilepath)
    # Abre el archivo de imagen actual utilizando Pillow (PIL).

    plt.imshow(targetimage)
    plt.title(targetfilename)  # Agrega el nombre del archivo como título
    plt.show()
    # Muestra la imagen cargada utilizando Matplotlib.


A continuación, llame la operación `search_faces_by_image` y vea si obtiene una coincidencia.

In [ ]:
threshold = 70
maxFaces = 5  # Permite que se devuelvan varias coincidencias.

# Iterar sobre todas las imágenes en la carpeta "images"
for targetfilename in image_files:
    targetfilepath = os.path.join(images_folder, targetfilename)
    print(f"\nProcesando la imagen: {targetfilename}")

    targetimage = Image.open(targetfilepath)
    imgWidth, imgHeight = targetimage.size

    with open(targetfilepath, 'rb') as timage:
        response2 = client.search_faces_by_image(
            CollectionId=collection_id,
            Image={'Bytes': timage.read()},
            FaceMatchThreshold=threshold,
            MaxFaces=maxFaces
        )

    faceMatches = response2.get('FaceMatches', [])

    if not faceMatches:
        print(f"No se encontraron coincidencias para la imagen: {targetfilename}")
    else:
        # Agrupar resultados por ExternalImageId
        grouped_matches = {}

        for match in faceMatches:
            external_id = match['Face']['ExternalImageId']
            similarity = match['Similarity']

            # Mantén solo la coincidencia con mayor similitud para cada ExternalImageId
            if external_id not in grouped_matches or similarity > grouped_matches[external_id]['Similarity']:
                grouped_matches[external_id] = {
                    'FaceId': match['Face']['FaceId'],
                    'Similarity': similarity,
                    'BoundingBox': match['Face']['BoundingBox']
                }

        # Dibujar los cuadros para las coincidencias agrupadas
        print('Coincidencias agrupadas por ExternalImageId:')
        draw = ImageDraw.Draw(targetimage)

        for external_id, data in grouped_matches.items():
            print(f"  ExternalImageId: {external_id}")
            print(f"    FaceId: {data['FaceId']}")
            print(f"    Similarity: {data['Similarity']:.2f}%")


## Tarea 7: dibujar un cuadro delimitador alrededor del rostro descubierto

Dibuje un cuadro delimitador alrededor del rostro descubierto.

In [ ]:
# Iterar sobre todas las imágenes en la carpeta
for targetfilename in image_files:
    targetfilepath = os.path.join(images_folder, targetfilename)
    print(f"\nProcesando la imagen: {targetfilename}")

    targetimage = Image.open(targetfilepath)
    imgWidth, imgHeight = targetimage.size

    with open(targetfilepath, 'rb') as timage:
        response2 = client.search_faces_by_image(
            CollectionId=collection_id,
            Image={'Bytes': timage.read()},
            FaceMatchThreshold=threshold,
            MaxFaces=maxFaces
        )

    faceMatches = response2.get('FaceMatches', [])

    if 'SearchedFaceBoundingBox' in response2:
        box = response2['SearchedFaceBoundingBox']

        draw = ImageDraw.Draw(targetimage)
        left = imgWidth * box['Left']
        top = imgHeight * box['Top']
        width = imgWidth * box['Width']
        height = imgHeight * box['Height']
        points = ((left, top), (left + width, top), (left + width, top + height), (left, top + height), (left, top))
        draw.line(points, fill='#00d400', width=15)

        print(f"Se encontró un rostro en {targetfilename}. Cuadro dibujado.")

        # Agrupar coincidencias por ExternalImageId
        grouped_matches = {}
        for match in faceMatches:
            external_id = match['Face']['ExternalImageId']
            if external_id not in grouped_matches:
                grouped_matches[external_id] = []
            grouped_matches[external_id].append({
                'FaceId': match['Face']['FaceId'],
                'Similarity': match['Similarity']
            })

        # Mostrar los resultados agrupados
        print("\nCoincidencias agrupadas por ExternalImageId:")
        for external_id, matches in grouped_matches.items():
            print(f"ExternalImageId: {external_id}")
            for match in matches:
                print(f"  - FaceId: {match['FaceId']}, Similarity: {match['Similarity']:.2f}%")

        plt.imshow(targetimage)
        plt.title(targetfilename)
        plt.show()
    else:
        print(f"No se encontraron rostros en la imagen: {targetfilename}")


## Tarea 8: eliminar la colección

Cuando finalice, elimine la colección. Para ello, ejecute el siguiente código: 


In [ ]:
print('Attempting to delete collection ' + collection_id)
# Imprime un mensaje indicando que se intentará eliminar la colección especificada.

status_code = 0
# Inicializa la variable para almacenar el código de estado de la operación.

try:
    # Bloque try para intentar eliminar la colección y manejar posibles errores.
    response = client.delete_collection(CollectionId=collection_id)
    # Llama a la API `delete_collection` para eliminar la colección con el ID especificado.
    status_code = response['StatusCode']
    # Obtiene el código de estado de la respuesta para confirmar el resultado.
    print('All done!')
    # Imprime un mensaje indicando que la colección se eliminó exitosamente.
    print(status_code)
    # Imprime el código de estado para confirmar la operación.

except ClientError as e:
    # Maneja posibles errores ocurridos durante la eliminación de la colección.
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        # Verifica si el error indica que la colección no fue encontrada.
        print('The collection ' + collection_id + ' was not found ')
        # Imprime un mensaje indicando que la colección no existe.
    else:
        # Maneja otros errores que no sean de tipo "colección no encontrada".
        print('Error other than Not Found occurred: ' + e.response['Error']['Message'])
        # Imprime el mensaje del error devuelto por AWS Rekognition.
    status_code = e.response['ResponseMetadata']['HTTPStatusCode']
    # Establece el código de estado según el error devuelto.


# ¡Felicitaciones!

Completó este laboratorio y ahora puede finalizarlo siguiendo las instrucciones en la guía del laboratorio.